In [2]:
import optuna
import neuron
from neuron import h
import matplotlib.pyplot as plt
from neuron.units import ms, mV
import numpy as np
import pickle
import os
from functools import partial

from scipy.signal import find_peaks
from scipy.optimize import curve_fit
from scipy import signal
# Load custom modules
pth_mod = '/Users/katharinaduecker/Documents/projects_brown/hnn-tuning/local_hnn/hnn_core/mod'
import sys
sys.path.append('/Users/katharinaduecker/Documents/projects_brown/hnn-tuning/new_cells_dev/utils')
from cell_builder_utils import l2_params, _get_dends, _get_pyr_soma, Cell
from iascone_mechs_utils import Mechs, get_iascone_params, Ih_condu_apical, distribute_gbar
from hay_mechs_utils import linear_gbar

In [21]:
# Load human data
pth_data = '/Users/katharinaduecker/Documents/projects_brown/hnn-tuning/data/l23'
with open(os.path.join(pth_data, 'ephys_l2.pkl'), 'rb') as f:
    l2_allen = pickle.load(f)
with open(os.path.join(pth_data, 'ephys_l3.pkl'), 'rb') as f:
    l3_allen = pickle.load(f)

# Combine L2 and L3 data for target ranges
target_median = {}
measures = ['RMP', 'sag', 'mean_isi', 'first_isi', 'upstroke_downstroke', 'ap_width', 'ap_threshold', 'FR_slope', 'tau']

for measure in measures:
    data = np.concatenate((l2_allen[measure].dropna().values,l3_allen[measure].dropna().values))
    target_median[measure] = np.median(data)

# target_ranges['attenuation'] = (0.0, 30.0)  # attenuation is in percent
# target_ranges['upstroke'] = (0.0, 20.0)  # upstroke is in mV/ms

# get the hand-tuned ca spike
import pickle
with open(os.path.join('/Users/katharinaduecker/Documents/projects_brown/hnn_dev/figures/newl3_iascone/', 'l23_ca_spike.pkl'), 'rb') as f:
    ca_properties = pickle.load(f)

for key in ca_properties.keys():
    target_median[key] = ca_properties[key]

In [39]:
# define cell model
class L23PyramidalCellModel():
    def __init__(self, name='L23_pyr_cell'):
        self.pth_mod = pth_mod
        self.mech_class = Mechs(self.pth_mod)
        
        # Get default parameters
        self.section_params, self.end_pts, self.cell_tree = l2_params()
        self.section_names = list(self.end_pts.keys())
        
        # Voltage initialization
        self.v_init = {'soma': -68.85159703362191,
                        'basal_1': -68.9013799898162,
                        'basal_2': -69.04833422207797,
                        'basal_3': -69.04833422207797,
                        'apical_oblique': -68.67137672693937,
                        'apical_trunk': -68.69214528583284,
                        'apical_1': -67.42523827628489,
                        'apical_tuft': -65.21723613840336}
        
        # Hand-tuned mechanism parameters
        self.default_params = {'soma': {'NaTs2_t_32d': 20_400e-4*3.5,
                                'SKv3_1': 6_930e-4*4.5,
                                'Nap_Et2': 1e-4/2,
                                'Ca_HVA': 9.92e-04,
                                'Ca_LVAst': 34.3e-4,
                                
                                #'SK_E2': 0.099433/2*0.001,
                                'pas': 1.0 / 35000.0,
                                'Ih': 0.000080*0.5,
                                'Im': 8e-3},
                        'basal': {'NaTs2_t_32d': 0.008009*0.01,#partial(linear_gbar,0.008009/2,0,255),
                                  'SKv3_1': 0.000513*0.01,#partial(linear_gbar,0,6_930e-4*0.1,255),
                                  'Ih': 0.00008,
                                  'pas':  1.0 / 35000.0},
                        'dend': {'NaTa_t_32d': 0.008009*0.1,
                                'SKv3_1': 0.000513*0.1,
                                'Ca_HVA': partial(linear_gbar, 2.78e-5/2*1., 2.78e-5/2*1, 400, hotzone=[400, 600], hotzone_factor=1),
                                'Ca_LVAst': partial(linear_gbar, 93.5e-6, 93.5e-6*1, 400, hotzone=[400, 600], hotzone_factor=1),
                                'pas': 1.0 / 35000.0,
                                #'SK_E2': 0.099433/2*0.00001,
                                'Ih': {'gbar': 0.000080, 'offset': -0.869600, 'slope':2, 'factor_exp':1/110},
                                'Im': 8e-3*0},
                        'e_pas': -75}
        
        self.sections = None
        self.cell = None
        self.build_cell()

    # build cell
    def build_cell(self, params=None):
        """Build the cell with given parameters or defaults"""
        if params is None:
            params = self.default_params
            
        # Build sections
        self.sections = _get_dends(self.section_params, 'L2Pyr', self.section_names, 
                                  self.end_pts, v_init=self.v_init)
        self.sections['soma'] = _get_pyr_soma(self.section_params, 'L2Pyr', 
                                             self.end_pts['soma'], v_init=self.v_init['soma'])
        
        # Create cell
        self.cell = Cell(self.sections, self.cell_tree, cell_type='L2_pyramidal', gid=0)
        self.cell.build()
        
        # Add mechanisms
        self.cell = self.mech_class.add_mechs(self.cell, params)
        
        return self.cell
    
    # run protocol function
    def run_protocol(self, protocol, param_values):
        """Run a protocol and return responses"""
        # Update parameters
        self.build_cell(param_values)

        if protocol.name == 'FR':
            return self.run_FR(protocol)
        elif protocol.name == 'attenuation':
            return self.run_attenuation(protocol)
        else:
            return self.run_stim(protocol)
        
    # template simulation function
    def run_stim(self, protocol):
        """Run dendritic stimulation protocol"""
        # Implementation of dendritic stimulation
        responses = {}
        
        # Set up stimulation
        stim = h.IClamp(self.cell._nrn_sections[protocol.section](protocol.seg))
        stim.delay = protocol.stim_delay
        stim.dur = protocol.stim_dur
        stim.amp = protocol.stim_amp
        
        # Set up recording
        t = h.Vector().record(h._ref_t)
        v = h.Vector().record(self.cell._nrn_sections[protocol.section](protocol.seg)._ref_v)
        ca = h.Vector().record(self.cell._nrn_sections[protocol.section](protocol.seg)._ref_cai)
        
        # Run simulation
        h.celsius = 37
        h.finitialize()
        h.continuerun(protocol.total_dur * ms)
        
        responses['time'] = np.array(t)
        responses[f'{protocol.section}_v'] = np.array(v)
        responses[f'{protocol.section}_ca'] = np.array(ca)
    
        return responses
    
    # RMP
    def calculate_RMP(self, responses):
        """ Get RMP """
        return responses['soma_v'][-3]

class StimProtocol:
    def __init__(self, name, stim_delay, stim_dur, stim_amp, total_dur, section, seg):
        self.name = name
        self.stim_delay = stim_delay
        self.stim_dur = stim_dur
        self.stim_amp = stim_amp
        self.total_dur = total_dur
        self.section = section
        self.seg = seg

In [40]:
RMP_protocol = StimProtocol(
    name='RMP',
    stim_delay=0 * ms,
    stim_dur=0 * ms,
    stim_amp=0,
    total_dur=200 * ms,
    section='soma',
    seg=0.5
)

In [46]:
def objective(trial):

    l23cell = L23PyramidalCellModel()

    ca_hva_gbar = trial.suggest_float('ca_hva_gbar', 0, 2e-3)
    ca_hva_slope = trial.suggest_float('ca_hva_slope', 0, 10)
    ca_hva_hotfact = trial.suggest_float('ca_hva_hotfact', 0, 20)

    ca_lva_gbar = trial.suggest_float('ca_lva_gbar', 0, 2e-3)
    ca_lva_slope = trial.suggest_float('ca_lva_slope', 0, 10)
    ca_lva_hotfact = trial.suggest_float('ca_lva_hotfact', 0, 20)
    
    suggest_params = {'soma': {'NaTs2_t_32d': trial.suggest_float('NaTs2_t_32d', 0, 8),
                                    'SKv3_1': trial.suggest_float('SKv3_1', 0, 5),
                                    'Nap_Et2': trial.suggest_float('Nap_Et2', 0, 1),
                                    'Ca_HVA': trial.suggest_float('Ca_HVA', 0, 1),
                                    'Ca_LVAst': trial.suggest_float('Ca_LVAst', 0, 1),
                                    'pas': trial.suggest_float('pas', 0, 1e-3),
                                    'Ih': trial.suggest_float('Ih', 0, 1e-4),
                                    'Im': trial.suggest_float('Im', 0, 1e-2)},
                  'basal': {'NaTs2_t_32d': trial.suggest_float('NaTs2_t_32d_basal', 0, 2e-3),
                                    'SKv3_1': trial.suggest_float('SKv3_1_basal', 0, 2e-3),
                                    'Ih': trial.suggest_float('Ih_basal', 0, 1e-3),
                                    'pas':  trial.suggest_float('pas_basal', 0, 1/9000)},
                            'dend': {'NaTa_t_32d': trial.suggest_float('NaTa_t_32d_apical', 0, 1e-2),
                                    'SKv3_1': trial.suggest_float('SKv3_1_apical', 0, 1e-2),
                                    'Ca_HVA': partial(linear_gbar, ca_hva_gbar, ca_hva_gbar*ca_hva_slope, 400, hotzone=[400, 600], hotzone_factor=ca_hva_hotfact),
                                    'Ca_LVAst': partial(linear_gbar, ca_lva_gbar, ca_lva_gbar*ca_lva_slope, 400, hotzone=[400, 600], hotzone_factor=ca_lva_hotfact),
                                    'pas': trial.suggest_float('pas_apical', 0, 1/9000.0),
                                    'SK_E2': trial.suggest_float('SK_E2_apical', 0, 1e-4),
                                    'Ih': {'gbar': trial.suggest_float('Ih_gbar', 0, 1e-2), 'offset': -0.869600, 'slope':2, 'factor_exp':1/110},
                                    'Im': trial.suggest_float('Im_apical', 0, 1e-2)},
                            'e_pas': -75}

    responses = l23cell.run_protocol(RMP_protocol, suggest_params)

    rmp = l23cell.calculate_RMP(responses)
    loss = (rmp - target_median['RMP']) ** 2
    return loss

In [48]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=1000)

[I 2025-08-22 16:21:51,927] A new study created in memory with name: no-name-a1be53ac-0ff3-493a-b25a-5f521030e1f9
[I 2025-08-22 16:21:52,093] Trial 0 finished with value: 4043.921391684655 and parameters: {'ca_hva_gbar': 0.00011402304450864565, 'ca_hva_slope': 9.953607597004316, 'ca_hva_hotfact': 5.954574151533086, 'ca_lva_gbar': 0.001510484580627924, 'ca_lva_slope': 3.4199685464245198, 'ca_lva_hotfact': 16.126522608910932, 'NaTs2_t_32d': 4.537763658443415, 'SKv3_1': 3.04014690529689, 'Nap_Et2': 0.15695071444520614, 'Ca_HVA': 0.8609108611146628, 'Ca_LVAst': 0.6886765668149181, 'pas': 0.0006928974726725224, 'Ih': 2.318551643048855e-05, 'Im': 0.00901384630953162, 'NaTs2_t_32d_basal': 0.0019344488330349768, 'SKv3_1_basal': 0.00024890514663530915, 'Ih_basal': 0.0003419295491446776, 'pas_basal': 9.309068246858302e-05, 'NaTa_t_32d_apical': 0.009077275568865542, 'SKv3_1_apical': 0.003900320261882815, 'pas_apical': 8.564908582395979e-05, 'SK_E2_apical': 2.578363438987946e-05, 'Ih_gbar': 0.0015

Mechanisms already loaded from path: /Users/katharinaduecker/Documents/projects_brown/hnn-tuning/local_hnn/hnn_core/mod.  Aborting.
Mechanisms already loaded from path: /Users/katharinaduecker/Documents/projects_brown/hnn-tuning/local_hnn/hnn_core/mod.  Aborting.
Mechanisms already loaded from path: /Users/katharinaduecker/Documents/projects_brown/hnn-tuning/local_hnn/hnn_core/mod.  Aborting.


[I 2025-08-22 16:21:52,341] Trial 2 finished with value: 62.63248687693983 and parameters: {'ca_hva_gbar': 0.0007738453081501842, 'ca_hva_slope': 2.032311848564421, 'ca_hva_hotfact': 11.97416785007325, 'ca_lva_gbar': 0.0010001132496574273, 'ca_lva_slope': 0.6297731089784786, 'ca_lva_hotfact': 9.94760063954239, 'NaTs2_t_32d': 7.039732919607893, 'SKv3_1': 4.425272520422954, 'Nap_Et2': 0.03586946912543221, 'Ca_HVA': 0.8214476801726701, 'Ca_LVAst': 0.5115745662690675, 'pas': 0.0007779065805709798, 'Ih': 4.99652674882759e-05, 'Im': 0.009105284584181248, 'NaTs2_t_32d_basal': 0.0010614770332375472, 'SKv3_1_basal': 0.00033396492402044367, 'Ih_basal': 0.00018214868328635247, 'pas_basal': 9.612048896018249e-05, 'NaTa_t_32d_apical': 0.0006426049889375718, 'SKv3_1_apical': 0.006104738567351462, 'pas_apical': 2.4656734232519377e-05, 'SK_E2_apical': 1.8517146700745104e-05, 'Ih_gbar': 0.005072427384925874, 'Im_apical': 0.008747424381888198}. Best is trial 2 with value: 62.63248687693983.
[I 2025-08-2

Mechanisms already loaded from path: /Users/katharinaduecker/Documents/projects_brown/hnn-tuning/local_hnn/hnn_core/mod.  Aborting.
Mechanisms already loaded from path: /Users/katharinaduecker/Documents/projects_brown/hnn-tuning/local_hnn/hnn_core/mod.  Aborting.
Mechanisms already loaded from path: /Users/katharinaduecker/Documents/projects_brown/hnn-tuning/local_hnn/hnn_core/mod.  Aborting.


[I 2025-08-22 16:21:52,719] Trial 5 finished with value: 4263.058276593675 and parameters: {'ca_hva_gbar': 0.0004544745156176948, 'ca_hva_slope': 1.2161747573264792, 'ca_hva_hotfact': 1.296275446005204, 'ca_lva_gbar': 0.0016891095384052902, 'ca_lva_slope': 5.570262661684483, 'ca_lva_hotfact': 11.91428265649966, 'NaTs2_t_32d': 5.788296643408836, 'SKv3_1': 3.634656840745511, 'Nap_Et2': 0.4237734294577813, 'Ca_HVA': 0.32200999388163276, 'Ca_LVAst': 0.6146861311294352, 'pas': 9.016679539364891e-05, 'Ih': 6.843789451601863e-05, 'Im': 0.004380882990832827, 'NaTs2_t_32d_basal': 0.0006571790840017768, 'SKv3_1_basal': 0.0006083310002383258, 'Ih_basal': 0.00028030166819334126, 'pas_basal': 7.873313111701278e-05, 'NaTa_t_32d_apical': 0.008498899178541445, 'SKv3_1_apical': 0.005921544047009968, 'pas_apical': 2.1465612869252613e-05, 'SK_E2_apical': 3.114678581203556e-05, 'Ih_gbar': 0.006529706361319903, 'Im_apical': 0.009982103486416285}. Best is trial 2 with value: 62.63248687693983.
[I 2025-08-22

Mechanisms already loaded from path: /Users/katharinaduecker/Documents/projects_brown/hnn-tuning/local_hnn/hnn_core/mod.  Aborting.
Mechanisms already loaded from path: /Users/katharinaduecker/Documents/projects_brown/hnn-tuning/local_hnn/hnn_core/mod.  Aborting.
Mechanisms already loaded from path: /Users/katharinaduecker/Documents/projects_brown/hnn-tuning/local_hnn/hnn_core/mod.  Aborting.


... (249 outputs left out)

Mechanisms already loaded from path: /Users/katharinaduecker/Documents/projects_brown/hnn-tuning/local_hnn/hnn_core/mod.  Aborting.
Mechanisms already loaded from path: /Users/katharinaduecker/Documents/projects_brown/hnn-tuning/local_hnn/hnn_core/mod.  Aborting.


[I 2025-08-22 16:22:29,536] Trial 262 finished with value: 6147.617867251896 and parameters: {'ca_hva_gbar': 0.0009134884608138232, 'ca_hva_slope': 5.19730498652327, 'ca_hva_hotfact': 1.9648087487009678, 'ca_lva_gbar': 0.0006653548309724353, 'ca_lva_slope': 0.5226372856204727, 'ca_lva_hotfact': 3.509440607692114, 'NaTs2_t_32d': 4.548281545481528, 'SKv3_1': 0.9426941606009496, 'Nap_Et2': 0.5585598666624904, 'Ca_HVA': 0.8866135894019804, 'Ca_LVAst': 0.47721087677455876, 'pas': 0.0005729052165477982, 'Ih': 2.6200385779250836e-06, 'Im': 0.0036973427090155045, 'NaTs2_t_32d_basal': 0.0009642920803142931, 'SKv3_1_basal': 0.00136452035385538, 'Ih_basal': 0.0007921235162883717, 'pas_basal': 3.0968496578189754e-05, 'NaTa_t_32d_apical': 0.004413820284330119, 'SKv3_1_apical': 0.0010083629445648706, 'pas_apical': 0.00010232918264191708, 'SK_E2_apical': 7.444509215646765e-05, 'Ih_gbar': 0.0005194287260472355, 'Im_apical': 0.0091785385244163}. Best is trial 241 with value: 0.002878120814273844.
[W 20

Mechanisms already loaded from path: /Users/katharinaduecker/Documents/projects_brown/hnn-tuning/local_hnn/hnn_core/mod.  Aborting.


KeyboardInterrupt: 